In [5]:
import OrcFxAPI
from pathlib import Path
import numpy as np

owd_file = Path(r"C:\Users\verav\Desktop\Studie\Afstuderen\PHASE2_ORCA\OW_data\Merganser_final.owd")
owr_file = Path(r"C:\Users\verav\Desktop\Studie\Afstuderen\PHASE2_ORCA\OW_results\Merganser_final_finer.owr")
xlsx_file = Path(r"C:\Users\verav\Desktop\Studie\Afstuderen\PHASE2_ORCA\XLSX_results\Merganser_final_finer.xlsx")


In [6]:
water_depth = 30.0  # m
wave_periods_coarse = [1.1,1.2,1.3,1.4,1.5,1.6,1.7,1.8,1.9, 2.0, 2.1, 2.2, 2.3, 2.4, 2.5, 2.6, 2.7, 2.8, 2.9]
  # jouw coarse

wave_periods_fine = np.arange(1, 8.01, 0.05)

wave_periods_rest = [8.1, 8.2, 8.3, 8.4, 8.5, 8.6, 8.7, 8.8, 8.9, 9.0, 9.5, 10.0, 10.5, 11, 11.5, 12, 13.5, 14, 14.5, 15, 15.5, 16, 17, 18, 19, 20 ]  # jouw rest

wave_periods = np.concatenate([
    # wave_periods_coarse,
    wave_periods_fine,
    wave_periods_rest
])
wave_headings = [0.0, 45.0, 90.0, 180.0]
   # deg

mass = 43  # ton, alleen goed als jouw model ook ton verwacht
com_x = 0.0
com_y = 0.0
com_z = 8.18
# voorbeeld inertia matrix rond CoM
Ixx = 3289 
Iyy = 3334
Izz = 5409
Ixy = 0.0
Ixz = 0.0
Iyz = 0.0


# =========================
# HELPER FUNCTIONS
# =========================
def set_value(obj, name, value):
    """
    Zet een OrcaWave data item.
    De exacte data-itemnamen moet je uit OrcaWave halen met F7.
    """
    try:
        obj[name] = value
        print(f"Set: {name} = {value}")
    except Exception as e:
        print(f"Kon data item niet zetten: {name}")
        print(f"  Waarde: {value}")
        print(f"  Fout: {e}")


def print_validation(diff):
    info = getattr(diff, "ValidationInformationText", "")
    warnings = getattr(diff, "ValidationWarningText", "")
    errors = getattr(diff, "ValidationErrorText", "")

    print("\n=== VALIDATION INFO ===")
    print(info if info else "(geen info)")

    print("\n=== VALIDATION WARNINGS ===")
    print(warnings if warnings else "(geen warnings)")

    print("\n=== VALIDATION ERRORS ===")
    print(errors if errors else "(geen errors)")

    return errors


# =========================
# MAIN
# =========================
diff = OrcFxAPI.Diffraction()
diff.LoadData(str(owd_file))

# -------------------------------------------------
# BELANGRIJK:
# De namen hieronder zijn PLACEHOLDERS / typische structuur.
# Vervang ze met de exacte OrcaWave data names via F7 in de GUI.
# -------------------------------------------------

# Environment
diff.SetData("WaterDepth", 0, 30.0)

# Wave periods
# Vaak moet je eerst een count zetten en daarna de tabel vullen
diff.SetData("NumberOfPeriodsOrFrequencies", 0, len(wave_periods))
for i, T in enumerate(wave_periods):
    diff.SetData(f"PeriodOrFrequency", i, T)

# Wave headings
diff.SetData("NumberOfWaveHeadings", 0, len(wave_headings))
for i, hdg in enumerate(wave_headings):
    diff.SetData(f"WaveHeading", i, hdg)


diff.SetData("BodyInertiaSpecifiedBy", 0, "Matrix (for a general body)")
diff.SetData("BodyInertiaTensorOriginType", 0, "Centre of mass")
# Body inertia / mass properties
# Let op: body index / naam kan anders zijn in jouw model
diff.SetData("BodyCentreOfMassX", 0, com_x)
diff.SetData("BodyCentreOfMassY", 0, com_y)
diff.SetData("BodyCentreOfMassZ", 0, com_z)

diff.SetData("BodyMass", 0, mass)



diff.SetData("BodyInertiaTensorRx", 0, Ixx)   # xx
diff.SetData("BodyInertiaTensorRy", 1, Iyy)   # yy
diff.SetData("BodyInertiaTensorRz", 2, Izz)   # zz


print("Rx:", diff.BodyInertiaTensorRx[0])
print("Ry:", diff.BodyInertiaTensorRy[1])
print("Rz:", diff.BodyInertiaTensorRz[2])


# Optioneel: output settings
# Ook hier weer: exacte namen via F7
# set_value(diff, "CalculateDisplacementRAOs", "Yes")
# set_value(diff, "CalculateLoadRAOs", "Yes")
# set_value(diff, "CalculateAddedMassAndDamping", "Yes")
# set_value(diff, "CalculateWaveDriftQTFs", "No")

# Validation
errors = print_validation(diff)
if errors:
    raise RuntimeError("Model heeft validation errors. Fix eerst de data-itemnamen of invoer.")



Rx: 3289.0
Ry: 3334.0
Rz: 5409.0

=== VALIDATION INFO ===
('Estimated peak memory required during calculation: 517 MiB per thread.',)

=== VALIDATION WARNINGS ===
('Calculation mesh: the following panels have large aspect ratio, consider re-meshing: 577 594 611 628 645 662 679 696 713 730 747 764 781 798 815 832 849 866 883 900 917 934 951 968 985 1002 1019 1036 1053 1070 1087 1104 1697 1714 1731 1748 1765 1782 1799 1816 1833 1850 1867 1884 1901 1918 1935 1952 1969 1986 2003 2020 2037 2054 2071 2088 2105 2122 2139 2156 2173 2190 2207 2224 2817 2834 2851 2868 2885 2902 2919 2936 2953 2970 2987 3004 3021 3038 3055 3072 3089 3106 3123 3140 3157 3174 3191 3208 3225 3242 3259 3276 3293 3310 3327 3344', 'Calculation mesh: the following panels are large compared to the wavelength of the shortest wave: 2 3 8-17 20 21 26-35 38 39 44-53 56 57 62-71 74 75 80-89 92 93 98-107 110 111 116-125 128 129 134-143 146 147 152-161 164 165 170-179 182 183 188-197 200 201 206-215 218 219 224-233 236 237 242-

In [7]:
# Run calculation
print("\n=== START CALCULATION ===")
diff.Calculate()
print("=== CALCULATION DONE ===")

# Save outputs
diff.SaveResults(str(owr_file))
diff.SaveResultsSpreadsheet(str(xlsx_file))

print("\nBestanden opgeslagen:")
print(f"  Results: {owr_file}")
print(f"  Spreadsheet: {xlsx_file}")


=== START CALCULATION ===
=== CALCULATION DONE ===

Bestanden opgeslagen:
  Results: C:\Users\verav\Desktop\Studie\Afstuderen\PHASE2_ORCA\OW_results\Merganser_final_finer.owr
  Spreadsheet: C:\Users\verav\Desktop\Studie\Afstuderen\PHASE2_ORCA\XLSX_results\Merganser_final_finer.xlsx
